# GWAS Association Analysis

**plink2 --glm firth · gwaslab · 1000 Genomes East Asian sample dataset**

This notebook documents association testing — the core GWAS step: testing each genetic
variant individually for association with a phenotype, while controlling for population
structure using the principal components from `01_PCA_1000Genomes.ipynb`.

> **Dataset note:** this uses the GWASTutorial repo's built-in sample dataset (`1kgeas`,
> ~495 individuals of East Asian ancestry) with a **synthetic, simulated binary phenotype**
> (`B1`) created for teaching purposes — not a real disease. The methodology shown here is
> the real GWAS workflow; the phenotype itself is not biologically meaningful.


## 1. The Regression Model

For each of **1,128,732 variants**, `plink2 --glm firth` fits a logistic regression:

```
phenotype ~ genotype + PC1 + PC2 + PC3 + PC4 + PC5
```

- **genotype**: additive coding (0, 1, or 2 copies of the tested allele)
- **PC1–PC5**: principal components from the PCA step, included as covariates to absorb
  ancestry-related confounding — without this, variants that merely differ in frequency
  between ancestry groups would falsely appear associated with the phenotype
- **Firth's correction**: used instead of standard logistic regression because it's more
  numerically stable for rare variants and small/imbalanced sample sizes, where standard
  logistic regression can fail to converge

```bash
plink2 \
    --bfile sample_data.clean \
    --pheno 1kgeas_binary.txt --pheno-name B1 \
    --maf 0.01 \
    --covar plink_results_projected.sscore --covar-col-nums 6-10 \
    --glm hide-covar firth firth-residualize single-prec-cc \
    --out 1kgeas
```

> **Problem faced:** the script's output paths assume it's run from inside the original
> `GWASTutorial/06_Association_tests/` folder (it references `../04_Data_QC/`,
> `../01_Dataset/`, etc. relative to the repo root). Running it from a different directory
> fails silently on missing input files. **Fix:** ran it in place, then copied the output
> file into this project's `results/` folder afterward.


## 2. Output Columns

| Column | Meaning |
|---|---|
| `#CHROM`, `POS`, `ID` | Variant location and identifier |
| `REF`, `ALT` | Reference and alternate alleles |
| `A1` | The tested ("effect") allele |
| `A1_FREQ` | Frequency of A1 in the sample |
| `TEST` | Genetic model — `ADD` = additive (each copy of A1 adds effect) |
| `OBS_CT` | Sample size used for that variant |
| `OR` | Odds ratio — effect size (OR > 1 = A1 associated with higher odds of the phenotype) |
| `LOG(OR)_SE` | Standard error of log(OR) |
| `Z_STAT` | Test statistic |
| `P` | P-value — the key number for significance |
| `ERRCODE` | `.` = no error; anything else flags a problem with that variant |


In [ ]:
import pandas as pd

sumstats = pd.read_csv("../results/association/1kgeas.B1.glm.firth", sep="\t")
print(f"Total variants tested: {len(sumstats):,}")
sumstats.head()

## 3. Manhattan and QQ Plots

Generated using [`gwaslab`](https://cloufield.github.io/gwaslab/), which reads PLINK2's
`--glm` output format directly and produces both plots together.

> **Problem faced:** installing `gwaslab` downgraded `pandas` (3.0.5 → 2.3.3) and
> `matplotlib` (3.11.1 → 3.8.4) in the shared virtual environment, since `gwaslab` pins
> older versions. This didn't break anything already built, but is worth knowing if
> re-running earlier notebooks afterward shows different plot styling.


In [ ]:
import gwaslab as gl

mysumstats = gl.Sumstats("../results/association/1kgeas.B1.glm.firth", fmt="plink2")
mysumstats.basic_check()

mysumstats.plot_mqq(
    skip=3,                 # skip variants with -log10(P) < 3 for plotting speed
    save="../results/association/mqq_plot.png",
    save_kwargs={"dpi": 300}
)

![Manhattan and QQ plot](../results/association/mqq_plot.png)

### Interpreting the Manhattan plot (left)

Each point is one variant; x-axis is genomic position by chromosome, y-axis is
-log10(P) — higher points are more significant. The dashed horizontal line marks the
standard genome-wide significance threshold (P = 5×10⁻⁸).

**Four loci cross genome-wide significance**: chromosomes **1, 2, 7, and 20** — with
chromosome 2 showing the strongest signal (-log10(P) ≈ 15).

### Interpreting the QQ plot (right)

Plots observed vs. expected -log10(P) under the null hypothesis (no association anywhere).
If most variants follow the diagonal and only a few deviate at the top, that's the expected
pattern for a real, localized signal — as opposed to genome-wide inflation, which would show
deviation starting early and across the whole distribution.

**λGC (genomic inflation factor) = 0.9884** — very close to the ideal value of 1.0,
indicating **no systematic inflation** from population stratification or cryptic
relatedness. This is a direct payoff from the PCA step: including PC1–PC5 as covariates
successfully absorbed ancestry-related confounding that would otherwise inflate results
genome-wide.


## 4. Key Takeaways

- Four loci (chr1, chr2, chr7, chr20) show genome-wide significant association with the
  (synthetic) phenotype, with chromosome 2 showing the strongest signal.
- λGC = 0.9884 confirms the PCA-derived covariates adequately controlled for population
  stratification — no genome-wide inflation.
- Firth's correction was necessary given the modest sample size (~495 people) to keep
  the regression numerically stable, especially for lower-frequency variants.

## Next Steps — Post-GWAS Analysis

- **LD Score Regression (LDSC)**: estimate SNP-based heritability and further confirm the
  absence of confounding, using the LDSC intercept alongside λGC
- **Variant annotation**: map the four significant loci to nearby genes and functional
  consequences, to interpret what they might mean biologically
